In [ ]:
%%sql -r CreateAgent
USE ROLE LOGISTICS_HOL_ADMIN;
USE DATABASE LOGISTICS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE HOL_WH;

CREATE OR REPLACE AGENT LOGISTICS_HOL.AGENTS.LOGISTICS_AGENT
WITH PROFILE = '{"display_name": "Logistics Operations Agent"}'
COMMENT = 'Orchestrates Cortex Analyst (shipment metrics) and Cortex Search (incident reports).'
FROM SPECIFICATION $$
models:
  orchestration: claude-sonnet-4-5
instructions:
  response: |
    You are a logistics operations analyst. Answer questions about carrier
    delivery performance. When the user asks about rates, counts, delays, or
    costs, use the shipment_analytics tool. When the user asks WHY deliveries
    were late or wants details from incident reports, use the incident_search
    tool. When a question needs both a number and an explanation, use both tools
    and combine them into one clear answer. Always cite the carrier and, when
    relevant, the incident type or region.
  orchestration: |
    Prefer shipment_analytics for any quantitative metric (on-time rate, late
    count, average delay, cost). Prefer incident_search for root-cause,
    narrative, or customer-sentiment questions. For "which carrier is worst and
    why" style questions, call shipment_analytics first to identify the carrier,
    then incident_search to explain it.
  sample_questions:
    - question: "Which carrier has the worst on-time delivery rate?"
    - question: "Why is DHL delivering late? Summarize the incident reports."
    - question: "Which carrier is underperforming and what are customers saying about it?"
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "shipment_analytics"
      description: "Structured shipment metrics: on-time rates, late counts, average delay days, and cost, sliced by carrier, region, customer segment, service type, and quarter."
  - tool_spec:
      type: "cortex_search"
      name: "incident_search"
      description: "Unstructured delivery incident reports explaining WHY shipments were delayed: mechanical failures, facility congestion, driver errors, weather, and customer complaints."
tool_resources:
  shipment_analytics:
    semantic_view: "LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV"
    execution_environment:
      type: warehouse
      warehouse: "HOL_WH"
  incident_search:
    name: "LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC"
    id_column: "INCIDENT_ID"
    title_column: "INCIDENT_TYPE"
$$;

SELECT 'Agent LOGISTICS_AGENT created' AS STATUS;

In [ ]:

%%sql -r TestAgent
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'LOGISTICS_HOL.AGENTS.LOGISTICS_AGENT',
  '{
    "messages": [
      {"role": "user", "content": [
        {"type": "text", "text": "Which carrier has the worst on-time delivery rate, and why? Summarize the incident reports."}
      ]}
    ],
    "stream": false
  }'
) AS AGENT_RESPONSE;

In [ ]:
%%sql -r CreateMCPServer
USE ROLE LOGISTICS_HOL_ADMIN;
USE SCHEMA LOGISTICS_HOL.AGENTS;

-- Use a plain unquoted name so the REST URL path resolves cleanly.
-- (Hyphens belong in the account hostname, not the MCP server object name.)
--
-- We expose the proven three-tool pattern so the MCP client can
-- orchestrate a full analytical workflow:
--   1) shipment_analytics  (CORTEX_ANALYST_MESSAGE) -> turns NL into SQL
--   2) query_executor      (SYSTEM_EXECUTE_SQL)     -> RUNS that SQL, returns rows
--   3) incident_search     (CORTEX_SEARCH_SERVICE_QUERY) -> returns report text
-- IMPORTANT: the Analyst tool returns *generated SQL*, not data. Without a
-- SYSTEM_EXECUTE_SQL tool the client has no way to run it, so always pair them.
CREATE OR REPLACE MCP SERVER LOGISTICS_HOL.AGENTS.LOGISTICS_MCP
FROM SPECIFICATION $$
tools:
  - name: "shipment_analytics"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV"
    description: "Converts natural-language questions about shipment metrics (on-time rate, late counts, average delay, cost by carrier/region/segment/service/quarter) into SQL. Returns a SQL query - run it with query_executor to get the numbers."
    title: "Shipment Analytics"
  - name: "incident_search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC"
    description: "Searches unstructured delivery incident reports explaining WHY shipments were delayed: mechanical failures, facility congestion, driver errors, weather, and customer complaints. Returns matching report text directly."
    title: "Delivery Incident Search"
  - name: "query_executor"
    type: "SYSTEM_EXECUTE_SQL"
    description: "Executes a SQL query against Snowflake and returns the result rows. Use this to run SQL produced by the shipment_analytics tool."
    title: "SQL Execution Tool"
$$;

SELECT 'MCP server LOGISTICS_MCP created with 3 tools' AS STATUS;
     

In [ ]:

%%sql -r ShowMCP
SHOW MCP SERVERS IN SCHEMA LOGISTICS_HOL.AGENTS;
DESCRIBE MCP SERVER LOGISTICS_HOL.AGENTS.LOGISTICS_MCP;
     

In [ ]:
%%sql -r CreateOAuthIntegration
USE ROLE ACCOUNTADMIN;

-- One OAuth integration is shared by all users; each authenticates individually.
-- ChatGPT's callback is usually CONNECTOR-SPECIFIC (https://chatgpt.com/connector/oauth/).
-- Start with the generic callback below, then after you create the connector in ChatGPT,
-- copy its exact callback URL and add it (see Step 7) via:
--   ALTER SECURITY INTEGRATION LOGISTICS_MCP_OAUTH
--     SET OAUTH_ALTERNATE_REDIRECT_URIS = ('https://chatgpt.com/connector/oauth/');
CREATE OR REPLACE SECURITY INTEGRATION LOGISTICS_MCP_OAUTH
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  ENABLED = TRUE
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://chatgpt.com/connector_platform_oauth_redirect'
  OAUTH_USE_SECONDARY_ROLES = IMPLICIT
  OAUTH_ISSUE_REFRESH_TOKENS = TRUE
  OAUTH_REFRESH_TOKEN_VALIDITY = 86400;

SELECT 'Security integration LOGISTICS_MCP_OAUTH created' AS STATUS;

In [ ]:
%%sql -r AttachIntegrationNetworkPolicy
USE ROLE ACCOUNTADMIN;

-- HOL reliability setting: allow remote MCP clients through this OAuth integration.
-- This does NOT replace your account-wide network policy; it scopes access to LOGISTICS_MCP_OAUTH.
-- For production, replace 0.0.0.0/0 with approved provider egress ranges.
CREATE OR REPLACE NETWORK POLICY MCP_OAUTH_ALLOW
  ALLOWED_IP_LIST = ('0.0.0.0/0')
  COMMENT = 'HOL: allow remote MCP client egress for the Cortex MCP OAuth integration';

ALTER SECURITY INTEGRATION LOGISTICS_MCP_OAUTH SET NETWORK_POLICY = MCP_OAUTH_ALLOW;

SELECT 'Integration-scoped network policy attached' AS STATUS;

In [ ]:
%%sql -r SetDefaultRole
-- MCP OAuth sessions use the connecting user's DEFAULT_ROLE (secondary roles are NOT used).
-- The user must also have a DEFAULT_WAREHOUSE or the session fails to initialize.
USE ROLE ACCOUNTADMIN;
SET my_user = CURRENT_USER();
ALTER USER IDENTIFIER($my_user) SET DEFAULT_ROLE = 'LOGISTICS_HOL_ADMIN' DEFAULT_WAREHOUSE = 'HOL_WH';

SELECT 'DEFAULT_ROLE and DEFAULT_WAREHOUSE set for ' || $my_user AS STATUS;

In [ ]:
%%sql -r GetOAuthSecrets
-- Copy OAUTH_CLIENT_ID and OAUTH_CLIENT_SECRET from this output into ChatGPT's custom connector.
-- Integration name is case-sensitive and must be UPPERCASE.
SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('LOGISTICS_MCP_OAUTH') AS OAUTH_SECRETS;

In [ ]:
%%sql -r GetMCPServerURL
-- Builds your exact MCP Server URL. Paste this into ChatGPT's custom connector.
-- Path segments are lowercase (Snowflake's REST convention); the host locator is
-- hyphenated (underscores in the host can break some MCP clients).
SELECT
  'https://'
  || REPLACE(LOWER(CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME()), '_', '-')
  || '.snowflakecomputing.com'
  || '/api/v2/databases/logistics_hol/schemas/agents/mcp-servers/logistics_mcp'
  AS MCP_SERVER_URL;

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

-- Update OAuth redirect for Claude
ALTER SECURITY INTEGRATION LOGISTICS_MCP_OAUTH
  SET OAUTH_REDIRECT_URI = 'https://claude.ai/api/mcp/auth_callback';

-- Update network policy to include Claude's egress IPs
ALTER NETWORK POLICY MCP_OAUTH_ALLOW
  SET ALLOWED_IP_LIST = ('0.0.0.0/0');

SELECT 'OAuth updated for Claude' AS STATUS;